# Script for Gemma2
Using Ollama (0.32.1) on local PC

## Imports

In [0]:
import re
import os
import pandas as pd
from  sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import KFold
from ollama import Client

## Loading MAUDE Data and preprocessing


In [0]:
# Data cleaning function - combined
def standardization(sent: str) -> str:
    '''
    Input: raw reviews (string)
    Output: cleaned & standardized reviews (string)
    '''
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    sent = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', sent.lower())  
    # Remove specific MAUDE patterns
    sent = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', sent)
    # Replace multiple spaces and strip leading/trailing whitespaces
    sent = re.sub(r'\s+', ' ', sent).strip()  
    
    return sent
 
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Input: DataFrame with column 'text' 
    Output: Cleaned DataFrame with standardized text
    '''
    # Standardize text by applying the standardization function to each row
    df["text"] = df["text"].apply(standardization) 
 
    return df
 
# params
max_words            = 2000
 
## Load and preprocess dataset
filename = "data/cybersecurity_annotated_data.pq"

df = pd.read_parquet(filename)
df = clean_dataframe(df[['ID', 'text', 'label']].dropna())
df['text'] =df['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))
df.head()

## Split data

Splitting the data in 5 chunks as used in the "traditional" models for foldwise comparability

In [0]:
#run the prompt only on the same sets as the traditional models in order to allow for comparison with the test scores of SVM, gemma
X = df["text"]
y = df["label"]
# Set up 5-folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []
FoldsDict = {}
for n, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    thisFold = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    FoldsDict[n] = thisFold

 

Selecting the fold

In [0]:
Fold = 0  #<- sets the current fold 0-4

x = FoldsDict[Fold]["X_test"]
y = FoldsDict[Fold]["y_test"] 
print(x)

## Setting up Ollama

Please enter the **host** specific to your environment
Enter into your command window with ollama installed: ollama run gemma2:9b

In [0]:
#start a local instance of ollama

client = Client(
  host='http://localhost:11434', #maybe your host is different
  headers={'Header': 'Cybersecurity Incident Classification Expert'}
)

model = "gemma2:9b" ##in your command window with ollama installed: ollama run gemma2:9b 


## Prompt examples
Zero shot and few shot examples.

#### Zero shot prompt


In [0]:
PromptVersion = "ZeroShot"
sys_prompt = """Answer format:
Respond ONLY with either '0' or '1'.
Respond with '0' if the report is not cybersecurity relevant.\
Respond with '1' if the report is cybersecurity relevant.\
"""

#### Two example prompts (few shot)
The Prompt name states the order in which the classes are described to the LLM. Run just one of the two

C0C1L = first the negative Class (Class 0), second the positive Class (Class 1); 

C1C0L = first the positive Class (Class 1), second the negative Class (Class 0)

In [0]:
PromptVersion = "Prompt2C0C1L"
sys_prompt = """
Your role is Cybersecurity incident classification expert.
You classify whether the report describes a cybersecurity incident case based on the following instructions:

    Classify as '0' if the report contains software updates, system backup modes, or general malfunctions.
    Classify as '0' if the report describes primarily a medical issue or surgical procedure.
    Classify as '0' if the report contains information about a general system error or malfunction.
    Classify as '0' if the report does not mention cybersecurity-related concerns.
    Classify as '0' if the report relates to a defective device unrelated to cybersecurity.
    Classify as '0' if the report describes fraudulent behaviour by businesses or by companies.
    Classify as '0' if the report lists many possible root causes.

    Classify as '1' if the report mentions actions like hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    Classify as '1' if the report states unauthorized access to systems, devices, networks, or data.
    Classify as '1' if the report contains info about loss or theft of sensitive information.
    Classify as '1' if the report mentions lacking cybersecurity protection such as firewalls and antivirus software.
    Classify as '1' if the patient suspects cybersecurity attacks, even if unverified.
    Classify as '1' if the report describes fraudulent behaviour related to networks, internet or computers.
    Classify as '1' if the report mentions switched off firewalls.
    Classify as '1' if the report content appears technically not plausible.

    Do not base the classification on the word "hacking" if it relates to coughing or when describing a mechanical issue.
    Respond ONLY with '0' or '1'.
    You do not provide an explanation.
    You do not provide a summary.

"""

In [0]:
PromptVersion = "Prompt2C1C0L"
sys_prompt= """
Your role is Cybersecurity incident classification expert.
You classify whether the report describes a cybersecurity incident case based on the following instructions:

    Classify as '1' if the report mentions actions like hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    Classify as '1' if the report states unauthorized access to systems, devices, networks, or data.
    Classify as '1' if the report contains info about loss or theft of sensitive information.
    Classify as '1' if the report mentions lacking cybersecurity protection such as firewalls and antivirus software.
    Classify as '1' if the patient suspects cybersecurity attacks, even if unverified.
    Classify as '1' if the report describes fraudulent behaviour related to networks, internet or computers.
    Classify as '1' if the report mentions switched off firewalls.
    Classify as '1' if the report content appears technically not plausible.

    Classify as '0' if the report contains software updates, system backup modes, or general malfunctions.
    Classify as '0' if the report describes primarily a medical issue or surgical procedure.
    Classify as '0' if the report contains information about a general system error or malfunction.
    Classify as '0' if the report does not mention cybersecurity-related concerns.
    Classify as '0' if the report relates to a defective device unrelated to cybersecurity.
    Classify as '0' if the report describes fraudulent behaviour by businesses or by companies.
    Classify as '0' if the report lists many possible root causes.

    Do not base the classification on the word "hacking" if it relates to coughing or when describing a mechanical issue.
    Respond ONLY with '0' or '1'.
    You do not provide an explanation.
    You do not provide a summary.

"""

#### Human Prompt

In [0]:
human_prompt = """Please classify the following report into *relevant' for cybersecurity or *not relevant* for cybersecurity."""

## Classification
**Note:** Make sure that the desired sys_prompt is used. It is the systemprompt cell that was run last   

In [0]:
#++++++++++++++  This sets the prompt cell that was run last as system prompt
used_sys_prompt = sys_prompt

RES_DF = pd.DataFrame()
for n,case in enumerate(x):
    print(f"This is number {n}")
    response =  client.chat(
                            model=model,
                            messages=[{"role":"system", "content":used_sys_prompt},
                                      {"role":"user", "content": human_prompt + case}])

 
    prediction = response.message.content

    
    try:
         prediction = int(prediction) 
    except:
        print(case)
        print(prediction)
        prediction = 2 #

    if (prediction < 0)  or (prediction > 2):
        prediction = 3

    print(str(prediction))
    RES_DF = RES_DF._append({"report": case, "label":y.iloc[n], "predict": prediction},ignore_index=True)

#save to excel

RES_DF.to_excel( PromptVersion + "Fold_" + str(Fold) + ".xlsx", index = False) #depends on the prompt cell which you ran

## Evaluate the results

In [0]:
def eval_model(y_test,y_pred):
    # value counts of the predicted labels
    print(RES_DF["predict"].value_counts())
  
    # Evaluate the model
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, digits= 3))

In [0]:
eval_model(RES_DF["label"].values ,RES_DF["predict"].values)